In [2]:
import re
import os

import sentencepiece as spm

## Load Data

In [3]:
with open('corpus/raw_ef.txt', 'w', encoding='utf-8') as corpus:
    for d, _, files in os.walk('corpus/datasets/endfield_data'):
        for file in files:
            path = os.path.join(d, file)
            if not path.endswith('.txt'):
                continue
            with open(path, 'r', encoding='utf-8') as file:
                corpus.write(file.read().strip() + '\n')

## Set Variables

In [4]:
CHARACTER_MAP = "gkamztlbdqiyfucxbhsjoprnweygtjmevchdxsanqolkrvwiypjzquhe"

def encode_line(text: str) -> str:
    """密文映射（保留换行/制表符原样输出）"""
    return "".join(CHARACTER_MAP[ord(c) % 56] if c not in "\n\t\r" else c for c in text)

SENTENCE_PATTERN = re.compile(r'([^。！？；\n]+[。！？；\n]+)')

out_src = 'corpus/endfield.skz'
out_tgt = 'corpus/endfield.zh'
raw = 'corpus/raw_ef.txt'
min_len = 20
max_len = 200

## Generate Corpus

In [5]:
os.makedirs(os.path.dirname(out_src) or '.', exist_ok=True)

valid_count = 0
with open(out_src, 'w', encoding='utf-8') as f_src, \
     open(out_tgt, 'w', encoding='utf-8') as f_tgt, \
     open(raw, 'r', encoding='utf-8') as f_raw:

    for line in f_raw:
        # 1. 切分句子
        sentences = SENTENCE_PATTERN.findall(line)
        for sent in sentences:
            # 2. 清洗多余空白（制表符/连续空格 -> 单空格）
            clean = re.sub(r'\s+', ' ', sent).strip()
            # 3. 长度过滤 & 写入平行语料
            if min_len <= len(clean) <= max_len:
                f_tgt.write(clean + '\n')          # 明文
                f_src.write(encode_line(clean) + '\n')  # 密文
                valid_count += 1

## Train Sarkaz Tokenizer

In [6]:
spm.SentencePieceTrainer.Train(
    input=out_src,                   # 密文文件
    model_prefix='models/sp_ef_skz',
    vocab_size=512,
    character_coverage=1.0,
    model_type='unigram',
    # input_sentence_size=2000000,       # 采样句数
    shuffle_input_sentence=True,
    split_by_whitespace=False,       # 密文无空格，关闭此切分
    # split_by_whitespace=True,
    split_digits=False
)
print("trained models: sp_ef_skz.model / sp_ef_skz.vocab")

trained models: sp_ef_skz.model / sp_ef_skz.vocab


## Train Chinese (Simplified) Tokenizer

In [8]:
spm.SentencePieceTrainer.Train(
    input=out_tgt,
    model_prefix="models/sp_ef_zh",
    vocab_size=16000,           # 中文推荐 8000~12000
    character_coverage=0.9995,  # 覆盖 99.9% 字符，生僻字自动走 <unk>
    model_type='unigram',
    # input_sentence_size=200000,
    shuffle_input_sentence=True,
    max_sentencepiece_length=16,
    num_threads=8
)
print("trained models: sp_ef_zh.model / sp_ef_zh.vocab")

trained models: sp_ef_zh.model / sp_ef_zh.vocab
